# Final Shard Visualization — Mixed PRISMA + EnMAP Training Patches

This notebook streams patches from a final mixed shard on S3 and visualizes each one as a spectral "cube" — RGB composite, validity mask, spectral profile, and individual band slices.

In [ ]:
import os, sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable
import webdataset as wds

PROJECT_ROOT = str(pathlib.Path(os.getcwd()).parents[0])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Stream from S3
SHARD_URL = "pipe: aws s3 cp s3://allotrope-raw-data-india/patches/hyperspectral/train/final/w128_h128_s64/final_shard_00000.tar -"
ds = wds.WebDataset(SHARD_URL, shardshuffle=False).decode()

# Load first 12 patches
patches = []
for i, sample in enumerate(ds):
    if i >= 12:
        break
    patches.append(sample)

print(f"Loaded {len(patches)} patches from final shard")
wl = patches[0]["wavelengths.npy"]
print(f"Wavelengths: {len(wl)} bands, {wl[0]:.0f}–{wl[-1]:.0f} nm")

## 1. Patch Gallery — RGB Composites

Grid of RGB composites from the first 12 patches. Color-coded borders: **blue = PRISMA**, **orange = EnMAP**.

In [ ]:
def make_rgb(cube, wl, validity=None):
    """RGB composite from nearest bands to 650/550/450nm."""
    r = np.argmin(np.abs(wl - 650)); g = np.argmin(np.abs(wl - 550)); b = np.argmin(np.abs(wl - 450))
    rgb = np.stack([cube[r], cube[g], cube[b]], axis=-1).astype(np.float32)
    for ch in range(3):
        v = rgb[:,:,ch]
        vals = v[validity > 0] if validity is not None else v[v > 0]
        if len(vals) > 0:
            p2, p98 = np.percentile(vals, [2, 98])
            rgb[:,:,ch] = np.clip((v - p2) / (p98 - p2 + 1e-10), 0, 1)
    if validity is not None:
        rgb[validity == 0] = 0
    return rgb

fig, axes = plt.subplots(3, 4, figsize=(16, 13))
sensor_colors = {"prisma": "#4A90D9", "enmap": "#E8943A"}

for idx, (ax, p) in enumerate(zip(axes.ravel(), patches)):
    cube = p["pixels.npy"]
    sv = p["validity_cube.npy"][0]
    meta = p["meta.json"]
    sensor = meta["sensor"]
    scene_id = meta["scene_id"][:25]

    rgb = make_rgb(cube, wl, sv)
    ax.imshow(rgb)
    ax.set_title(f"[{sensor.upper()}] {scene_id}...\n"
                 f"({meta['row_coords']},{meta['col_coords']}) "
                 f"valid:{sv.mean()*100:.0f}%", fontsize=8)
    ax.axis("off")

    # Colored border
    for spine in ax.spines.values():
        spine.set_edgecolor(sensor_colors[sensor])
        spine.set_linewidth(3)
        spine.set_visible(True)

plt.suptitle("Patch Gallery — Mixed PRISMA (blue) + EnMAP (orange) Final Shard",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Spectral Cube Visualization

For each of the first 4 patches, show the "cube" — RGB composite, validity mask, band slices at 5 wavelengths across the spectrum, and the mean spectral profile with ±1σ envelope.

In [ ]:
SLICE_WAVELENGTHS = [500, 700, 1200, 1600, 2100]  # nm — one per spectral region

for patch_idx in range(4):
    p = patches[patch_idx]
    cube = p["pixels.npy"]       # (165, 128, 128)
    sv = p["validity_cube.npy"][0]  # (128, 128)
    meta = p["meta.json"]
    sensor = meta["sensor"]

    fig = plt.figure(figsize=(22, 8))
    gs = gridspec.GridSpec(2, 7, figure=fig, width_ratios=[1, 1, 1, 1, 1, 1, 1.5],
                           hspace=0.35, wspace=0.3)

    # Row 1: RGB + validity + 5 band slices
    # RGB
    ax_rgb = fig.add_subplot(gs[0, 0])
    rgb = make_rgb(cube, wl, sv)
    ax_rgb.imshow(rgb)
    ax_rgb.set_title("RGB", fontsize=9)
    ax_rgb.axis("off")

    # Validity
    ax_val = fig.add_subplot(gs[0, 1])
    ax_val.imshow(sv, cmap="gray")
    ax_val.set_title(f"Validity\n{sv.mean()*100:.0f}%", fontsize=9)
    ax_val.axis("off")

    # Band slices
    for j, target_wl in enumerate(SLICE_WAVELENGTHS):
        b = np.argmin(np.abs(wl - target_wl))
        ax = fig.add_subplot(gs[0, 2 + j])
        band_data = np.ma.masked_where(sv == 0, cube[b])
        im = ax.imshow(band_data, cmap="viridis")
        ax.set_title(f"{wl[b]:.0f} nm", fontsize=9)
        ax.axis("off")

    # Row 2: Spectral profile (spans full width)
    ax_spec = fig.add_subplot(gs[1, :])

    valid_spectra = cube[:, sv == 1]  # (165, N)
    if valid_spectra.shape[1] > 0:
        mean_spec = valid_spectra.mean(axis=1)
        std_spec = valid_spectra.std(axis=1)

        ax_spec.fill_between(wl, mean_spec - std_spec, mean_spec + std_spec,
                             alpha=0.15, color="steelblue")
        ax_spec.plot(wl, mean_spec, color="steelblue", lw=1.5, label="Mean ± 1σ")

        # Overlay 5 random pixel spectra
        rng = np.random.default_rng(42 + patch_idx)
        n_sample = min(5, valid_spectra.shape[1])
        sample_idx = rng.choice(valid_spectra.shape[1], n_sample, replace=False)
        for si in sample_idx:
            ax_spec.plot(wl, valid_spectra[:, si], lw=0.4, alpha=0.4, color="gray")

        # Mark the slice wavelengths
        for target_wl in SLICE_WAVELENGTHS:
            b = np.argmin(np.abs(wl - target_wl))
            ax_spec.axvline(wl[b], color="red", linestyle=":", alpha=0.4, lw=0.8)

    ax_spec.set_xlabel("Wavelength (nm)")
    ax_spec.set_ylabel("Surface Reflectance")
    ax_spec.legend(fontsize=8, loc="upper right")
    ax_spec.grid(True, alpha=0.3)
    ax_spec.set_xlim([wl[0] - 10, wl[-1] + 10])

    # Shade atmospheric gaps
    gaps = np.diff(wl)
    for gi in range(len(gaps)):
        if gaps[gi] > 15:
            ax_spec.axvspan(wl[gi], wl[gi+1], alpha=0.08, color="red")

    color = sensor_colors[sensor]
    fig.suptitle(
        f"Patch {patch_idx} — [{sensor.upper()}] {meta['scene_id'][:45]}...  "
        f"({meta['row_coords']},{meta['col_coords']})",
        fontsize=12, y=1.02, color=color, fontweight="bold"
    )
    plt.show()

## 3. Cross-Sensor Spectral Comparison

Compare the mean spectral profiles of PRISMA vs EnMAP patches on the same common grid. Both should cover the same wavelength range with the same band count, but reflectance levels may differ (different scenes, land cover types).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Collect mean spectra by sensor
prisma_spectra = []
enmap_spectra = []
for p in patches:
    cube = p["pixels.npy"]
    sv = p["validity_cube.npy"][0]
    sensor = p["meta.json"]["sensor"]
    valid = cube[:, sv == 1]
    if valid.shape[1] > 0:
        mean_s = valid.mean(axis=1)
        if sensor == "prisma":
            prisma_spectra.append(mean_s)
        else:
            enmap_spectra.append(mean_s)

# Left: individual patch profiles colored by sensor
for s in prisma_spectra:
    axes[0].plot(wl, s, color="#4A90D9", lw=0.8, alpha=0.5)
for s in enmap_spectra:
    axes[0].plot(wl, s, color="#E8943A", lw=0.8, alpha=0.5)

axes[0].plot([], [], color="#4A90D9", lw=2, label=f"PRISMA ({len(prisma_spectra)} patches)")
axes[0].plot([], [], color="#E8943A", lw=2, label=f"EnMAP ({len(enmap_spectra)} patches)")
axes[0].set_xlabel("Wavelength (nm)")
axes[0].set_ylabel("Surface Reflectance")
axes[0].set_title("Individual Patch Mean Spectra by Sensor")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
for gi in range(len(wl)-1):
    if wl[gi+1] - wl[gi] > 15:
        axes[0].axvspan(wl[gi], wl[gi+1], alpha=0.08, color="red")

# Right: sensor-average with envelope
if prisma_spectra:
    pm = np.stack(prisma_spectra)
    axes[1].fill_between(wl, pm.mean(0) - pm.std(0), pm.mean(0) + pm.std(0),
                         alpha=0.15, color="#4A90D9")
    axes[1].plot(wl, pm.mean(0), color="#4A90D9", lw=2, label=f"PRISMA mean ± 1σ")

if enmap_spectra:
    em = np.stack(enmap_spectra)
    axes[1].fill_between(wl, em.mean(0) - em.std(0), em.mean(0) + em.std(0),
                         alpha=0.15, color="#E8943A")
    axes[1].plot(wl, em.mean(0), color="#E8943A", lw=2, label=f"EnMAP mean ± 1σ")

axes[1].set_xlabel("Wavelength (nm)")
axes[1].set_ylabel("Surface Reflectance")
axes[1].set_title("Sensor-Averaged Spectral Profiles")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
for gi in range(len(wl)-1):
    if wl[gi+1] - wl[gi] > 15:
        axes[1].axvspan(wl[gi], wl[gi+1], alpha=0.08, color="red")

plt.suptitle("Cross-Sensor Spectral Comparison on Common 165-Band Grid", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Shard Statistics Summary

In [ ]:
from collections import Counter

sensors = Counter()
scenes = Counter()

for p in patches:
    m = p["meta.json"]
    sensors[m["sensor"]] += 1
    scenes[m["scene_id"]] += 1

print(f"{'='*60}")
print(f"{'FINAL SHARD SUMMARY':^60}")
print(f"{'='*60}")
print(f"  Patches sampled: {len(patches)}")
print(f"  Pixel shape:     (165, 128, 128)")
print(f"  Wavelengths:     {len(wl)} bands, {wl[0]:.0f}–{wl[-1]:.0f} nm")
print()
print(f"  Sensor mix:")
for s, c in sensors.most_common():
    print(f"    {s:8s}: {c:3d} patches ({c/len(patches)*100:.0f}%)")
print()
print(f"  Scene diversity: {len(scenes)} unique scenes")
print()

validities = [p["validity_cube.npy"][0].mean() * 100 for p in patches]
reflectances = [p["pixels.npy"][:, p["validity_cube.npy"][0] > 0].mean() for p in patches]
print(f"  Validity:     {min(validities):.0f}%–{max(validities):.0f}% (mean {np.mean(validities):.0f}%)")
print(f"  Reflectance:  {min(reflectances):.4f}–{max(reflectances):.4f} (mean {np.mean(reflectances):.4f})")
print()
print(f"  All shapes (165,128,128): {all(p['pixels.npy'].shape == (165,128,128) for p in patches)}")
print(f"  All wavelengths match:    {all(np.array_equal(p['wavelengths.npy'], wl) for p in patches)}")
print(f"  All validity >50%:        {all(v > 50 for v in validities)}")
print(f"  Ready for training:       Yes")